In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

C:\Users\tjmja\AppData\Local\Temp\ipykernel_33896\2611793992.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[sql_query]
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [6]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='301996f5-884b-4815-a6f8-fc35824f1cf2'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'sql_query', 'arguments': '{"query": "SELECT ArtistId, Name FROM artists WHERE Name LIKE \'S%\'"}'}, '__gemini_function_call_thought_signatures__': {'call_661112': 'EnMKcQFpFH0Tu8BlawPwNBXLBXFkOqojpyzdzejahPNJQQMmYNYd1vJlUtcWMn+QWVq7Oi4VSxhuz9Nr0CZiWcSoEJ9vvBcRbrmq59unUTH5PMn8kAii5eAvtr5my9OfxNat1gU1IcYxcL+F4ELcA7MOEQOV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f68f-983f-7081-ad0a-7dff68f68489-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT ArtistId, Name FROM artists WHERE Name LIKE 'S%'"}, 'id': 'call_661112', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 

In [7]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT 
    ar.Name, 
    COUNT(il.InvoiceLineId) as TotalSold
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
JOIN Track t ON al.AlbumId = t.AlbumId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
WHERE ar.Name LIKE 'S%'
GROUP BY ar.ArtistId
ORDER BY TotalSold DESC
LIMIT 1;
